In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics.pairwise import cosine_similarity

df_clean = pd.read_csv("online_retail_day1_cleaned.csv")

df_clean["CustomerID"] = df_clean["CustomerID"].astype(int)

print("Cleaned dataset restored successfully!")
print("Dataset Shape:", df_clean.shape)

Cleaned dataset restored successfully!
Dataset Shape: (392692, 9)


In [11]:
# Create customer-product purchase matrix

customer_product_matrix = df_clean.pivot_table(
    index="CustomerID",
    columns="StockCode",
    values="Quantity",
    aggfunc="sum",
    fill_value=0
)

print("Customer-Product Matrix Created!")
print("Matrix Shape:", customer_product_matrix.shape)

customer_product_matrix.head()

Customer-Product Matrix Created!
Matrix Shape: (4338, 3665)


StockCode,10002,10080,10120,10123C,10124A,10124G,10125,10133,10135,11001,...,90214V,90214W,90214Y,90214Z,BANK CHARGES,C2,DOT,M,PADS,POST
CustomerID,,,,,,,,,,,,,,,,,,,,,
12346,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
12347,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
12348,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,9
12349,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
12350,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1


In [12]:
# Calculate similarity between products
product_similarity = cosine_similarity(
    customer_product_matrix.T
)

# Convert into DataFrame
product_similarity_df = pd.DataFrame(
    product_similarity,
    index=customer_product_matrix.columns,
    columns=customer_product_matrix.columns
)

print("Product Similarity Calculated Successfully!")
print("Similarity Matrix Shape:", product_similarity_df.shape)

Product Similarity Calculated Successfully!
Similarity Matrix Shape: (3665, 3665)


In [13]:
def recommend_products(product_code, top_n=5):

    if product_code not in product_similarity_df.columns:
        return "Product not found!"

    similar_products = product_similarity_df[product_code].sort_values(
        ascending=False
    )

    similar_products = similar_products.drop(product_code)

    return similar_products.head(top_n)

In [14]:
sample_product = customer_product_matrix.columns[0]

print("Selected Product:", sample_product)

recommendations = recommend_products(sample_product)

print("\nRecommended Similar Products:")
print(recommendations)

Selected Product: 10002

Recommended Similar Products:
StockCode
10125     0.853895
23224     0.713423
23222     0.699006
85014A    0.626153
20682     0.597314
Name: 10002, dtype: float64


In [15]:
# Get product names from the dataset
product_names = df_clean.groupby("StockCode")["Description"].agg(
    lambda x: x.mode().iloc[0] if not x.mode().empty else "Unknown Product"
)

# Select a sample product
sample_product = customer_product_matrix.columns[0]

# Get similar products
recommendations = recommend_products(sample_product)

# Create a readable recommendation table
recommendation_table = pd.DataFrame({
    "ProductCode": recommendations.index,
    "ProductName": recommendations.index.map(product_names),
    "SimilarityScore": recommendations.values
})

print("Selected Product:", product_names.get(sample_product, "Unknown Product"))
print("\nRecommended Similar Products:")
recommendation_table

Selected Product: INFLATABLE POLITICAL GLOBE 

Recommended Similar Products:


,ProductCode,ProductName,SimilarityScore
0,10125,MINI FUNKY DESIGN TAPES,0.853895
1,23224,CHERUB HEART DECORATION GOLD,0.713423
2,23222,CHRISTMAS TREE HANGING GOLD,0.699006
3,85014A,BLACK/BLUE POLKADOT UMBRELLA,0.626153
4,20682,RED RETROSPOT CHILDRENS UMBRELLA,0.597314


In [17]:
import pandas as pd
import numpy as np

# Load cleaned dataset
df_clean = pd.read_csv("online_retail_day1_cleaned.csv")
df_clean["InvoiceDate"] = pd.to_datetime(df_clean["InvoiceDate"])
df_clean["CustomerID"] = df_clean["CustomerID"].astype(int)

# Calculate reference date
reference_date = df_clean["InvoiceDate"].max() + pd.Timedelta(days=1)

# Create RFM table
rfm = df_clean.groupby("CustomerID").agg({
    "InvoiceDate": lambda x: (reference_date - x.max()).days,
    "InvoiceNo": "nunique",
    "TotalAmount": "sum"
})

rfm.rename(columns={
    "InvoiceDate": "Recency",
    "InvoiceNo": "Frequency",
    "TotalAmount": "Monetary"
}, inplace=True)

# Restore cluster numbers and segment names
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

rfm_log = np.log1p(rfm)

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_log)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm["Cluster"] = kmeans.fit_predict(rfm_scaled)

segment_names = {
    0: "Champions",
    1: "At-Risk Customers",
    2: "Recent Occasional Buyers",
    3: "Regular Customers"
}

rfm["Segment"] = rfm["Cluster"].map(segment_names)

print("RFM data restored successfully!")
print("Shape:", rfm.shape)

RFM data restored successfully!
Shape: (4338, 5)


In [18]:
segment_summary = rfm.groupby("Segment").agg({
    "Recency": "mean",
    "Frequency": "mean",
    "Monetary": "mean"
}).round(2)

segment_summary

,Recency,Frequency,Monetary
Segment,,,
At-Risk Customers,181.51,1.32,341.00
Champions,12.17,13.75,8088.02
Recent Occasional Buyers,17.70,2.19,557.32
Regular Customers,71.64,4.08,1801.78
